# Second Eye: manual test notebook

Tests the Second Eye fracture-detection API by hand: send an X-ray, see the calibrated answer.

> **Research prototype, not a medical device.** Don't use it on real patients.

**Start the API first** (in a terminal, from the repo root):

```bash
scripts/vision/run_local.sh            # full mode: custom answer options work (loads Qwen3-8B)
MODE=lean scripts/vision/run_local.sh  # faster start, built-in question only
scripts/vision/run_local.sh notebook   # or: start the API and open this notebook in one go
```

The notebook only talks to the API over HTTP, so it works the same against a remote deployment.
Set `SECOND_EYE_URL` and `SECOND_EYE_API_KEY` to point it elsewhere.

In [ ]:
import json, os, time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from PIL import Image

URL = os.environ.get("SECOND_EYE_URL", "http://127.0.0.1:8701")
API_KEY = os.environ.get("SECOND_EYE_API_KEY")          # only needed if the server was started with one
HEADERS = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}

# run from the repo root or from examples/
ROOT = Path.cwd() if (Path.cwd() / "splits").exists() else Path.cwd().parent
print("API:", URL, "| repo:", ROOT)

## 1. Is the API up?

In [ ]:
health = requests.get(f"{URL}/health", timeout=10).json()
health

- `device` is `mps` on a Mac, `cuda` on a GPU box.
- `custom_options: false` means the server runs in lean mode, so section 4 will be refused.

## 2. Helpers

In [ ]:
def predict(image_path, options=None, question=None):
    """POST one image to /predict; returns the JSON answer."""
    data = {}
    if options:
        data["options"] = json.dumps(options)
    if question:
        data["question"] = question
    with open(image_path, "rb") as f:
        r = requests.post(f"{URL}/predict", files={"image": f}, data=data, headers=HEADERS, timeout=600)
    if r.status_code != 200:
        raise RuntimeError(f"{r.status_code}: {r.text}")
    return r.json()


def show(image_path, answer, truth=None, ax=None):
    """The X-ray next to its answer distribution."""
    if ax is None:
        _, ax = plt.subplots(1, 2, figsize=(9, 3.6), gridspec_kw={"width_ratios": [1, 1.3]})
    img_ax, bar_ax = ax
    img_ax.imshow(Image.open(image_path).convert("L"), cmap="gray")
    img_ax.set_axis_off()
    title = Path(image_path).name + (f"\ntruth: {'fracture' if truth else 'no fracture'}" if truth is not None else "")
    img_ax.set_title(title, fontsize=9)
    opts, probs = list(answer["probabilities"]), list(answer["probabilities"].values())
    colors = ["#0b7a8c" if o == answer["decision"] else "#b8c4cc" for o in opts]
    bar_ax.barh(opts[::-1], probs[::-1], color=colors[::-1])
    bar_ax.set_xlim(0, 1)
    if answer.get("threshold") is not None:
        bar_ax.axvline(answer["threshold"], color="#c0392b", ls="--", lw=1,
                       label=f"decision threshold {answer['threshold']:.3f}")
        bar_ax.legend(loc="lower right", fontsize=8, frameon=False)
    kind = "calibrated" if answer["calibrated"] else "raw softmax (not calibrated)"
    bar_ax.set_title(f"decision: {answer['decision']}\n{kind} · {answer['latency_ms']:.0f} ms", fontsize=9)
    bar_ax.tick_params(labelsize=8)

## 3. Try a few FracAtlas images

These come from the **validation** split. The test split is kept untouched for the reported
results. If the dataset isn't downloaded (`scripts/vision/step2_data.sh`), skip to section 5 and
use your own image.

In [ ]:
val_csv = ROOT / "splits" / "val.csv"
have_data = val_csv.exists() and (ROOT / "data" / "FracAtlas").exists()
if have_data:
    val = pd.read_csv(val_csv)
    sample = pd.concat([val[val.label == 1].sample(3, random_state=1), val[val.label == 0].sample(3, random_state=1)])
    sample["path"] = sample.path.map(lambda p: str(ROOT / p))
    display(sample[["image_id", "label", "region", "view"]])
else:
    print("FracAtlas not found under", ROOT / "data", "- skip to section 5")

In [ ]:
if have_data:
    fig, axes = plt.subplots(len(sample), 2, figsize=(9, 3.3 * len(sample)),
                             gridspec_kw={"width_ratios": [1, 1.3]})
    for (_, row), ax in zip(sample.iterrows(), axes):
        show(row.path, predict(row.path), truth=row.label, ax=ax)
    plt.tight_layout()

How to read the answers:
- The decision uses the operating point chosen on val for **≥ 0.90 sensitivity**. It flags a
  fracture when calibrated p(fracture) ≥ the red threshold, which is low (about 0.07) on purpose:
  missing a fracture costs more than a false alarm.
- Expect some false alarms on the non-fractured images. On test this operating point has
  specificity 0.72.

## 4. Custom answer options (full mode only)

Any wording is embedded on the fly by CLM's action tower. The first call with a new option set
is slow on a 24 GB Mac, because Qwen3-8B has to load from swap. Repeat calls are cached.

Custom options come back as **raw, uncalibrated** probabilities: the calibration was fit on the
trained wording and does not transfer. They still rank images sensibly, as the paraphrase
ablation showed (AUROC 0.91–0.93).

In [ ]:
if have_data and health.get("custom_options"):
    img = sample.path.iloc[0]
    for opts in (["A fracture is present.", "No fracture is present."],
                 ["Yes, the bone is broken.", "No, the bone is intact."]):
        t = time.time()
        a = predict(img, options=opts)
        print(f"{opts}\n  -> {a['probabilities']}  decision: {a['decision']}  ({time.time() - t:.1f} s wall)")
elif not health.get("custom_options"):
    print("server is in lean mode; restart without MODE=lean to try custom options")

## 5. Your own image

In [ ]:
MY_IMAGE = None          # e.g. "/Users/you/Downloads/wrist.jpg"

if MY_IMAGE:
    answer = predict(MY_IMAGE)
    show(MY_IMAGE, answer)
    answer

## 6. Quick sanity check on 60 validation images

This is a smoke test, not an evaluation: it runs on val, the split the threshold was tuned on,
so the numbers are optimistic. The reported results are in `docs/VISION.md`.

In [ ]:
if have_data:
    from sklearn.metrics import roc_auc_score
    check = pd.concat([val[val.label == 1].sample(20, random_state=0), val[val.label == 0].sample(40, random_state=0)])
    rows = []
    for r in check.itertuples():
        a = predict(str(ROOT / r.path))
        p = a["probabilities"]["Yes, there is a fracture."]
        rows.append({"image_id": r.image_id, "label": r.label, "p_fracture": p,
                     "flagged": a["decision"].startswith("Yes"), "latency_ms": a["latency_ms"]})
    res = pd.DataFrame(rows)
    sens = res[res.label == 1].flagged.mean()
    spec = (~res[res.label == 0].flagged).mean()
    print(f"AUROC {roc_auc_score(res.label, res.p_fracture):.3f} | sensitivity {sens:.2f} | specificity {spec:.2f} "
          f"| p50 latency {res.latency_ms.median():.0f} ms (n={len(res)})")
    display(res.sort_values("p_fracture", ascending=False).head(10))

## 7. Error handling (optional)

The API refuses bad input with a clear message rather than crashing.

In [ ]:
def status(**kw):
    r = requests.post(f"{URL}/predict", headers=HEADERS, **kw)
    return r.status_code, r.json().get("detail")

readme = ROOT / "README.md"
print("not an image :", status(files={"image": open(readme, "rb")}))
if have_data:
    print("one option   :", status(files={"image": open(sample.path.iloc[0], "rb")}, data={"options": '["only one"]'}))